In [2]:
from alpaca.trading.client import TradingClient

API_KEY = "PKV2J2YQTIFPH3F4VBCMGFHQNZ"
SECRET_KEY = "2ntaDKy48njGGc3Ev6cEVLtBp4cy7bnZ917KW7mggBah"

client = TradingClient(API_KEY, SECRET_KEY, paper=True)
account = client.get_account()
print(f"Cash: ${account.cash}")
print(f"Portfolio value: ${account.portfolio_value}")

Cash: $100000
Portfolio value: $100000


In [3]:

import yfinance as yf
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from blackscholes import implied_vol, call_price

API_KEY = "PKV2J2YQTIFPH3F4VBCMGFHQNZ"
SECRET_KEY = "2ntaDKy48njGGc3Ev6cEVLtBp4cy7bnZ917KW7mggBah"

In [4]:
ticker = yf.Ticker("AAPL")
S = ticker.history(period='1d')['Close'].iloc[-1]  # current stock price
r = 0.05  # risk free rate

# Get nearest expiry options
expiry = ticker.options[2]  # third expiry date
chain = ticker.option_chain(expiry)
calls = chain.calls

# Filter for ATM options
calls = calls[(calls['strike'] > S * 0.95) & (calls['strike'] < S * 1.05)]
calls = calls[calls['lastPrice'] > 0.5]

print(f"Stock price: ${S:.2f}")
print(calls[['strike', 'lastPrice', 'impliedVolatility']].head())

Stock price: $312.06
    strike  lastPrice  impliedVolatility
30   297.5      15.21           0.311530
31   300.0      12.79           0.278083
32   302.5      10.52           0.264534
33   305.0       8.35           0.235970
34   307.5       6.26           0.231331


In [5]:
# Find the strike closest to current stock price
atm_idx = (calls['strike'] - S).abs().idxmin()
atm_call = calls.loc[atm_idx]

market_price = atm_call['lastPrice']
K = atm_call['strike']

# Calculate time to expiry
expiry_date = datetime.strptime(expiry, '%Y-%m-%d')
T = (expiry_date - datetime.now()).days / 365

# Calculate implied vol using our solver
iv = implied_vol(market_price, S, K, T, r, option_type='call')
print(f"Strike: ${K}")
print(f"Market price: ${market_price}")
print(f"Time to expiry: {T:.3f} years")
print(f"Implied volatility: {iv:.4f} ({iv*100:.2f}%)")

Strike: $312.5
Market price: $3.2
Time to expiry: 0.014 years
Implied volatility: 0.2272 (22.72%)


In [6]:
# Get 30 days of historical prices
hist = ticker.history(period='30d')['Close']

# Calculate 20-day historical volatility (annualised)
returns = np.log(hist / hist.shift(1)).dropna()
hist_vol = returns.std() * np.sqrt(252)

print(f"Today's implied vol: {iv*100:.2f}%")
print(f"20-day historical vol: {hist_vol*100:.2f}%")
print(f"IV/HV ratio: {iv/hist_vol:.2f}")

if iv > 1.5 * hist_vol:
    print("SIGNAL: VOL BREAKOUT — BUY")
elif iv < hist_vol:
    print("SIGNAL: VOL NORMALISED — SELL")
else:
    print("SIGNAL: HOLD")

Today's implied vol: 22.72%
20-day historical vol: 19.72%
IV/HV ratio: 1.15
SIGNAL: HOLD


In [7]:
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

client = TradingClient(API_KEY, SECRET_KEY, paper=True)

def execute_signal(signal, client, symbol='AAPL', qty=10):
    if signal == 'BUY':
        order = MarketOrderRequest(
            symbol=symbol,
            qty=qty,
            side=OrderSide.BUY,
            time_in_force=TimeInForce.GTC
        )
        client.submit_order(order)
        print(f"Bought {qty} shares of {symbol}")
    elif signal == 'SELL':
        order = MarketOrderRequest(
            symbol=symbol,
            qty=qty,
            side=OrderSide.SELL,
            time_in_force=TimeInForce.GTC
        )
        client.submit_order(order)
        print(f"Sold {qty} shares of {symbol}")
    else:
        print("No trade — holding")

# Determine signal
if iv > 1.5 * hist_vol:
    signal = 'BUY'
elif iv < hist_vol:
    signal = 'SELL'
else:
    signal = 'HOLD'

execute_signal(signal, client)


No trade — holding


In [8]:
def run_strategy(client, symbol='AAPL', qty=10):
    ticker = yf.Ticker(symbol)
    
    # Get current stock price
    S = ticker.history(period='1d')['Close'].iloc[-1]
    
    # Get ATM implied vol
    expiry = ticker.options[2]
    chain = ticker.option_chain(expiry)
    calls = chain.calls
    calls = calls[(calls['strike'] > S * 0.95) & (calls['strike'] < S * 1.05)]
    calls = calls[calls['lastPrice'] > 0.5]
    
    atm_idx = (calls['strike'] - S).abs().idxmin()
    atm_call = calls.loc[atm_idx]
    market_price = atm_call['lastPrice']
    K = atm_call['strike']
    expiry_date = datetime.strptime(expiry, '%Y-%m-%d')
    T = (expiry_date - datetime.now()).days / 365
    
    iv = implied_vol(market_price, S, K, T, r, option_type='call')
    
    # Get historical vol
    hist = ticker.history(period='30d')['Close']
    returns = np.log(hist / hist.shift(1)).dropna()
    hist_vol = returns.std() * np.sqrt(252)
    
    # Generate signal
    if iv > 1.5 * hist_vol:
        signal = 'BUY'
    elif iv < hist_vol:
        signal = 'SELL'
    else:
        signal = 'HOLD'
    
    print(f"Stock: ${S:.2f} | IV: {iv*100:.2f}% | HV: {hist_vol*100:.2f}% | Signal: {signal}")
    execute_signal(signal, client, symbol, qty)

run_strategy(client)

Stock: $312.06 | IV: 22.72% | HV: 19.72% | Signal: HOLD
No trade — holding
